## Dask Sentinel-2 cloudless

Before running the cells in this notebook, open a Terminal and run:

```bash
cd cloudless-mosaic
pip install -e .
```

In [ ]:
import os
from dask_gateway import Gateway
from cloudless_mosaic.app import main

## Dask Gateway connection

The environment variable `DASK_GATEWAY_ADDRESS` contains the Dask Gateway internal service address.

Create a `Gateway` object that will later be used to create a Dask cluster

In [ ]:
gateway_url = os.environ.get("DASK_GATEWAY_ADDRESS")

gateway = Gateway(gateway_url)



## Create a Dask cluster

The environment variable `DASK_IMAGE` contains the container image to use as for the Dask cluster workers.

In [ ]:
cluster_options = gateway.cluster_options()

image = os.environ.get("DASK_IMAGE", "ghcr.io/eoap/dask-app-package/cloudless-mosaic:2.0.0")
worker_cores = 0.5
worker_cores_limit = int(os.environ.get("DASK_WORKER_CORES_LIMIT", "1"))
worker_memory = int(os.environ.get("DASK_WORKER_MEMORY_GB", "2"))

cluster_options['image'] = image
cluster_options['worker_cores'] = worker_cores
cluster_options['worker_cores_limit'] = worker_cores_limit
cluster_options['worker_memory'] = f"{worker_memory} G"

In [ ]:
cluster = gateway.new_cluster(cluster_options)

Print the cluster name:


In [ ]:
cluster.name

Print the cluster dashboard link although it's not accessible via browser.

In [ ]:
cluster.dashboard_link

If there's a port forward on the `traefik-dask-gateway` service, the dashboard link becomes:

In [ ]:
print(f"https://localhost:8001/clusters/{cluster.name}/status")

Get the cluster client and scale the cluster workers:

In [ ]:

client = cluster.get_client()

cluster.adapt(minimum=2, maximum=int(os.environ.get("DASK_MAX_WORKERS", "2")))
client.wait_for_workers(2, timeout=120)

## Cloudless monthly mosaic generation

In [ ]:
params = {
    "start_date": "2020-10-01", 
    "end_date": "2020-12-31",
    "aoi": "-122.27508544921875,47.54687159892238,-121.96128845214844,47.745787772920934",
    "bands": ["nir", "red", "green"],
    "collection": "sentinel-2-l2a",
    "resolution": 100,
    "max_items": int(os.environ.get("DASK_LAB_MAX_ITEMS", "1000")), 
    "max_cloud_cover": 25
}

try:
    main(**params)
except Exception:
    cluster.shutdown()
    gateway.close()
    raise
from pathlib import Path
assert Path("catalog.json").is_file()


## Dispose the cluster

In [ ]:
client.close()
cluster.shutdown()
gateway.close()